<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0f0f 0%, #1a1a1a 100%); border: 2px solid #a855f7; border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(168, 85, 247, 0.2); font-family: sans-serif;'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ clonador de Voz Ilimitado</h1>
  <h3 style='color: #d8b4fe; margin: 0 0 15px 0; font-weight: 400;'><strong>By ROTA PROMPT AI</strong></h3>
  <div style='display: flex; justify-content: center; gap: 15px;'>
    <a href="http://youtube.com/@canalwfreitas" target="_blank" style="background: #FF0000; color: white; padding: 8px 16px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 14px;">▶ YouTube</a>
    <a href="https://www.instagram.com/0xwsf/" target="_blank" style="background: linear-gradient(45deg, #f09433 0%, #e6683c 25%, #dc2743 50%, #cc2366 75%, #bc1888 100%); color: white; padding: 8px 16px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 14px;">📸 Instagram</a>
  </div>
</div>

> ⚠️ Certifique-se de selecionar **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU** antes de rodar.

In [ ]:
#@title 🚀 INICIAR GERADOR (Clique aqui apenas uma vez)
import IPython.display as display
from IPython.display import clear_output
from IPython.utils import io
import os
import sys
import time

# ==========================================
# 1. TELA DE LOADING NO COLAB
# ==========================================
html_loading = """
<div style='background: #0a0812; padding: 40px; border-radius: 12px; text-align: center; font-family: "Inter", sans-serif; max-width: 650px; margin: 0 auto; border: 1px solid #1e1533;'>
  <div style="border: 3px solid #111; border-top: 3px solid #7c3aed; border-radius: 50%; width: 40px; height: 40px; animation: spin 1s linear infinite; margin: 0 auto 20px auto;"></div>
  <style>@keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }</style>
  <h2 style='color: white; margin: 0 0 10px 0; font-size: 1.5em; font-weight: 400;'>Preparando Ambiente...</h2>
</div>
"""
display.display(display.HTML(html_loading))

# ==========================================
# 2. BLOQUEIO DE CONSOLE E INSTALAÇÕES
# ==========================================
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

error_message = None
share_url = None

with io.capture_output() as captured:
    try:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers>=5.3"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        # Mantido a sua string de instalação original
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice", "pydub", "gradio==6.11.0"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        import torch
        import numpy as np
        from omnivoice import OmniVoice, OmniVoiceGenerationConfig
        import gradio as gr

        model = OmniVoice.from_pretrained(
            'k2-fsa/OmniVoice',
            device_map='cuda',
            dtype=torch.float16,
            load_asr=True,
            token=False,
        )
        sampling_rate = model.sampling_rate

        # ==========================================
        # 3. CSS APENAS PARA O DESIGN PREMIUM
        # ==========================================
        CSS = """
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap');
        * { font-family: 'Inter', sans-serif !important; }
        
        body, .gradio-container { 
            background-color: #0b0811 !important;
            background-image: radial-gradient(circle at 50% 0%, #1e1533 0%, #0b0811 50%) !important;
            color: #ffffff !important; 
        }
        
        .gradio-container .form { background: transparent !important; border: none !important; box-shadow: none !important; }
        .label-wrap { display: none !important; } 
        
        /* --- CAIXA DE TEXTO (Esquerda) --- */
        #caixa-texto {
            position: relative;
            background: #15121e !important;
            border-radius: 12px !important;
            overflow: hidden;
            box-shadow: 0 10px 30px rgba(0,0,0,0.5) !important;
            height: 100% !important;
        }
        #caixa-texto::before {
            content: ''; position: absolute; top: 0; left: 0; right: 0; height: 5px;
            background: linear-gradient(90deg, #4b6cb7 0%, #8b5cf6 100%);
            z-index: 10;
        }
        #caixa-texto textarea {
            background: transparent !important;
            border: none !important;
            color: #d1d5db !important;
            font-size: 16px !important;
            padding: 30px 25px !important;
            resize: none !important;
            box-shadow: none !important;
            min-height: 250px !important;
        }
        #caixa-texto textarea:focus { outline: none !important; border: none !important; }
        
        /* --- COLUNA DIREITA --- */
        #coluna-direita {
            display: flex !important;
            flex-direction: column !important;
            justify-content: space-between !important; 
            height: 100% !important;
        }
        
        /* Espreme a caixa nativa do gr.Audio para parecer um botão normal */
        #caixa-audio { border: none !important; background: transparent !important; margin-bottom: 10px !important; }
        #caixa-audio .upload-container {
            background: #0f0a17 !important;
            border: 1px solid #3c2a5c !important;
            border-radius: 4px !important;
            min-height: 55px !important;
            height: 55px !important;
            display: flex;
            align-items: center;
            justify-content: center;
            transition: all 0.2s;
        }
        #caixa-audio .upload-container:hover { background: #1a1229 !important; }
        
        /* Dropdown Linguagem */
        #caixa-linguagem .wrap {
            background: #0f0a17 !important;
            border: 1px solid #201533 !important;
            border-radius: 4px !important;
            color: #d1d5db !important;
        }
        
        /* Botão Gerar Áudio */
        #botao-gerar {
            background: #0f0a17 !important;
            border: 1px solid #201533 !important;
            border-radius: 4px !important;
            color: #d1d5db !important;
            font-weight: 400 !important;
            text-transform: uppercase;
            letter-spacing: 1px;
            padding: 15px !important;
            box-shadow: none !important;
            transition: all 0.2s;
            margin-top: 15px !important; 
            width: 100% !important;
        }
        #botao-gerar:hover { background: #1a1229 !important; border-color: #3c2a5c !important; }
        
        /* Botão Desativado (durante o loading) */
        #botao-gerar:disabled, button.primary:disabled {
            background: #1e1533 !important;
            border-color: #3c2a5c !important;
            color: #8b5cf6 !important;
            cursor: not-allowed !important;
            opacity: 0.8;
        }

        /* --- ÁUDIO RESULTADO (Base Pílula) --- */
        #caixa-resultado { margin-top: 20px !important; }
        #caixa-resultado .wrap, #caixa-resultado audio {
            background: #7b6196 !important;
            border-radius: 40px !important;
            border: none !important;
            padding: 5px 10px !important;
        }
        #caixa-resultado * { color: white !important; }
        """

        BRAND_HTML = """
        <div style="text-align: center; margin-bottom: 30px; margin-top: 20px;">
            <h1 style="color: white; margin: 0; font-size: 26px; font-weight: 500;">Gerador e Clonador de Audio</h1>
            <p style="color: #6d4b99; font-size: 15px; font-weight: 400; margin: 8px 0 0 0;">By Lucro A Lucro</p>
        </div>
        """

        # ── Language dropdown ──
        LANGUAGES = [
            "Auto", "English (en)", "Chinese (zh)", "Japanese (ja)", "Korean (ko)",
            "French (fr)", "German (de)", "Spanish (es)", "Portuguese (pt)",
            "Russian (ru)", "Arabic (ar)", "Hindi (hi)", "Italian (it)",
            "Dutch (nl)", "Turkish (tr)", "Polish (pl)", "Swedish (sv)",
            "Thai (th)", "Vietnamese (vi)", "Indonesian (id)", "Malay (ms)",
        ]

        def lang_dropdown():
            return gr.Dropdown(label="Language (optional)", choices=LANGUAGES, value="Auto")

        def gen_settings():
            with gr.Accordion("⚙️ Advanced Settings", open=False):
                ns = gr.Slider(8, 64, value=32, step=1, label="Inference Steps")
                gs = gr.Slider(0.0, 10.0, value=3.0, step=0.1, label="Guidance Scale")
                dn = gr.Slider(0.0, 1.0, value=0.8, step=0.05, label="Denoise Ratio")
                sp = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Speed")
                du = gr.Slider(0, 30, value=0, step=0.5, label="Duration (0 = auto)")
                pp = gr.Checkbox(value=True, label="Preprocess Prompt")
                po = gr.Checkbox(value=True, label="Postprocess Output")
            return ns, gs, dn, sp, du, pp, po

        # ── Voice Design categories ──
        CATEGORIES = {
            "Gender": ["male", "female"],
            "Age": ["child", "teenager", "young adult", "middle-aged", "elderly"],
            "Pitch": ["very low pitch", "low pitch", "moderate pitch", "high pitch", "very high pitch"],
            "Style": ["whisper"],
            "English Accent": ["american accent", "british accent", "australian accent",
                               "canadian accent", "indian accent", "chinese accent",
                               "japanese accent", "korean accent", "portuguese accent",
                               "russian accent"],
            "Chinese Dialect": ["四川话", "陕西话", "广东话", "东北话", "河南话",
                                "云南话", "贵州话", "桂林话", "济南话"],
        }

        ATTR_INFO = {
            "Gender": "Speaker gender",
            "Age": "Approximate speaker age",
            "Pitch": "Voice pitch level",
            "Style": "Speaking style",
            "English Accent": "English accent (effective for English text)",
            "Chinese Dialect": "Chinese dialect (effective for Chinese text)",
        }

        def build_instruct(groups):
            selected = [g for g in groups if g and g != "Auto"]
            return ", ".join(selected) if selected else ""

        # ── LÓGICA ORIGINAL INTACTA ──
        def generate_speech(text, language, ref_audio, instruct,
                            num_step, guidance_scale, denoise,
                            speed, duration, preprocess_prompt, postprocess_output,
                            mode="clone", ref_text=None):
            if not text or not text.strip():
                return None, "⚠️ Please enter some text."

            lang_code = None
            if language and language != "Auto":
                lang_code = language.split("(")[-1].rstrip(")").strip() if "(" in language else language

            gen_config = OmniVoiceGenerationConfig(
                num_step=int(num_step or 32),
                guidance_scale=float(guidance_scale) if guidance_scale is not None else 2.0,
                denoise=bool(denoise) if denoise is not None else True,
                preprocess_prompt=bool(preprocess_prompt),
                postprocess_output=bool(postprocess_output),
            )

            kw = dict(
                text=text.strip(),
                language=lang_code,
                generation_config=gen_config,
            )

            if speed is not None and float(speed) != 1.0:
                kw["speed"] = float(speed)
            if duration is not None and float(duration) > 0:
                kw["duration"] = float(duration)

            if mode == "clone":
                if ref_audio is None:
                    return None, "⚠️ Please upload a reference audio."
                kw["voice_clone_prompt"] = model.create_voice_clone_prompt(
                    ref_audio=ref_audio,
                    ref_text=ref_text,
                )

            if mode == "design":
                if instruct and instruct.strip():
                    kw["instruct"] = instruct.strip()

            try:
                audio = model.generate(**kw)
            except Exception as e:
                return None, f"❌ Error: {type(e).__name__}: {e}"

            waveform = audio[0].squeeze() 
            if hasattr(waveform, 'numpy'):
                waveform = waveform.numpy()
            waveform = (waveform * 32767).astype(np.int16)
            return (sampling_rate, waveform), f"✅ Generated {waveform.shape[-1]/sampling_rate:.1f}s audio at {sampling_rate}Hz"

        # ── INTERFACE GRÁFICA (Com IDs para o CSS Premium) ──
        with gr.Blocks(title="OmniVoice Demo", theme=gr.themes.Base(), css=CSS) as demo:
            gr.HTML(BRAND_HTML)

            with gr.Tabs():
                with gr.TabItem("🎤 Voice Clone"):
                    
                    with gr.Row(equal_height=True):
                        with gr.Column(scale=5):
                            vc_text = gr.Textbox(
                                show_label=False,
                                lines=9,
                                placeholder="Que app nativo para dispositivos móveis devemos criar?",
                                elem_id="caixa-texto"
                            )
                        with gr.Column(scale=2, elem_id="coluna-direita"):
                            
                            vc_ref_audio = gr.Audio(
                                label="📄 Carregar audio de referencia",
                                type="filepath", 
                                elem_id="caixa-audio"
                            )
                            
                            vc_lang = lang_dropdown()
                            vc_lang.elem_id = "caixa-linguagem"
                            
                            vc_btn = gr.Button("GERAR AUDIO", elem_id="botao-gerar")

                    with gr.Row():
                        with gr.Column(scale=1):
                            vc_audio = gr.Audio(
                                show_label=False, 
                                type="numpy", 
                                elem_id="caixa-resultado"
                            )

                    with gr.Accordion("⚙️ Status e Configurações Extras", open=False):
                        vc_ref_text = gr.Textbox(
                            label="Reference Text (optional)", lines=2,
                            placeholder="Transcript of ref audio. Leave empty for auto-transcription."
                        )
                        vc_ns, vc_gs, vc_dn, vc_sp, vc_du, vc_pp, vc_po = gen_settings()
                        vc_status = gr.Textbox(label="Status", lines=2)

                    def clone_fn(text, lang, ref_aud, ref_text, ns, gs, dn, sp, du, pp, po):
                        return generate_speech(text, lang, ref_aud, None, ns, gs, dn, sp, du, pp, po,
                                               mode="clone", ref_text=ref_text or None)

                    # --- LÓGICA DE LOADING VISUAL PARA ABA 1 ---
                    vc_btn.click(
                        fn=lambda: (gr.update(value="⏳ GERANDO...", interactive=False), "⏳ Processando áudio, aguarde..."),
                        inputs=None,
                        outputs=[vc_btn, vc_status]
                    ).then(
                        fn=clone_fn,
                        inputs=[vc_text, vc_lang, vc_ref_audio, vc_ref_text,
                                vc_ns, vc_gs, vc_dn, vc_sp, vc_du, vc_pp, vc_po],
                        outputs=[vc_audio, vc_status]
                    ).then(
                        fn=lambda: gr.update(value="GERAR AUDIO", interactive=True),
                        inputs=None,
                        outputs=[vc_btn]
                    )

                with gr.TabItem("🎨 Voice Design"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            vd_text = gr.Textbox(
                                label="Text to Synthesize", lines=4,
                                placeholder="Enter text here...")
                            vd_lang = lang_dropdown()
                            vd_groups = []
                            for cat, choices in CATEGORIES.items():
                                vd_groups.append(
                                    gr.Dropdown(label=cat, choices=["Auto"] + choices,
                                                value="Auto", info=ATTR_INFO.get(cat)))
                            vd_ns, vd_gs, vd_dn, vd_sp, vd_du, vd_pp, vd_po = gen_settings()
                            vd_btn = gr.Button("🔊 Generate", variant="primary", size="lg")
                        with gr.Column(scale=1):
                            vd_audio = gr.Audio(label="Output Audio", type="numpy")
                            vd_status = gr.Textbox(label="Status", lines=2)

                    def design_fn(text, lang, ns, gs, dn, sp, du, pp, po, *groups):
                        return generate_speech(text, lang, None, build_instruct(groups),
                                               ns, gs, dn, sp, du, pp, po, mode="design")

                    # --- LÓGICA DE LOADING VISUAL PARA ABA 2 ---
                    vd_btn.click(
                        fn=lambda: (gr.update(value="⏳ GERANDO...", interactive=False), "⏳ Processando áudio, aguarde..."),
                        inputs=None,
                        outputs=[vd_btn, vd_status]
                    ).then(
                        fn=design_fn,
                        inputs=[vd_text, vd_lang, vd_ns, vd_gs, vd_dn, vd_sp,
                                vd_du, vd_pp, vd_po] + vd_groups,
                        outputs=[vd_audio, vd_status]
                    ).then(
                        fn=lambda: gr.update(value="🔊 Generate", interactive=True),
                        inputs=None,
                        outputs=[vd_btn]
                    )

        # LANÇAMENTO OCULTO
        app, local_url, share_url = demo.queue().launch(share=True, inline=False, quiet=True)

    except Exception as e:
        import traceback
        error_message = traceback.format_exc()

# ==========================================
# 4. EXIBIR BOTÃO NO COLAB
# ==========================================
clear_output(wait=True) 

if share_url:
    html_btn = f"""
    <div style='background: #0a0812; padding: 40px; border-radius: 12px; text-align: center; font-family: "Inter", sans-serif; max-width: 650px; margin: 0 auto; border: 1px solid #1e1533;'>
      <h2 style='color: white; margin: 0 0 10px 0; font-size: 1.8em; font-weight: 500;'>✨ Sistema Online!</h2>
      <a href='{share_url}' target='_blank' style='background: #7b6196; color: #ffffff; padding: 15px 35px; border-radius: 6px; text-decoration: none; font-weight: bold; font-size: 16px; display: inline-block; margin-top: 15px; transition: opacity 0.2s;'>
        ABRIR GERADOR (NOVA ABA)
      </a>
      <p style='color: #444; font-size: 0.8em; margin: 25px 0 0 0;'>🟢 Célula em execução. Não feche ou pause o Colab.</p>
    </div>
    """
    display.display(display.HTML(html_btn))
    
    # Mantém a célula rodando para a GPU não desligar
    while True:
        time.sleep(1)
else:
    html_error = f"""
    <div style='text-align: center; padding: 30px; background: #1a0a0a; border: 1px solid #7f1d1d; border-radius: 12px; font-family: "Inter", sans-serif; max-width: 650px; margin: 0 auto;'>
      <h2 style='color: #ef4444; margin: 0 0 10px 0;'>⚠️ Erro ao carregar</h2>
      <div style='background: #000; padding: 10px; border-radius: 8px; text-align: left; overflow-x: auto;'>
        <code style='color: #fca5a5; font-size: 0.8em; white-space: pre-wrap;'>{error_message}</code>
      </div>
    </div>
    """
    display.display(display.HTML(html_error))